In [ ]:
#1.Install PySpark
!pip install pyspark

In [ ]:
#2.Import libraries
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import pandas as pd
from sklearn.datasets import load_iris

In [ ]:
#3.Start spark
spark=SparkSession.builder \
    .appName("IrisClassification") \
    .getOrCreate()

print("Successfully")
print("Spark version:",spark.version)

SparkSession successfully
Spark version: 4.0.2


In [ ]:
#4.Load Iris dataset
#sklearn has it built in so I just load from there
#then convert to Spark DataFrame
iris=load_iris()
pdf=pd.DataFrame(iris.data,columns=iris.feature_names)
pdf["species"]=iris.target_names[iris.target]

sdf=spark.createDataFrame(pdf)
print("First 5 rows:")
sdf.show(5)

print("Schema:")
sdf.printSchema()

print("Class count:")
sdf.groupBy("species").count().show()

First 5 rows:
+-----------------+----------------+-----------------+----------------+-------+
|sepal length (cm)|sepal width (cm)|petal length (cm)|petal width (cm)|species|
+-----------------+----------------+-----------------+----------------+-------+
|              5.1|             3.5|              1.4|             0.2| setosa|
|              4.9|             3.0|              1.4|             0.2| setosa|
|              4.7|             3.2|              1.3|             0.2| setosa|
|              4.6|             3.1|              1.5|             0.2| setosa|
|              5.0|             3.6|              1.4|             0.2| setosa|
+-----------------+----------------+-----------------+----------------+-------+
only showing top 5 rows
Schema:
root
 |-- sepal length (cm): double (nullable = true)
 |-- sepal width (cm): double (nullable = true)
 |-- petal length (cm): double (nullable = true)
 |-- petal width (cm): double (nullable = true)
 |-- species: string (nullable = tr

## Dataset Overview
150 rows, 4 features, 3 classes.

50 samples per class, perfectly balanced.

No class imbalance to deal with here.

| Species    | Count |
|------------|-------|
| versicolor | 50    |
| setosa     | 50    |
| virginica  | 50    |

In [ ]:
#5.Preprocessing
#Iris is usually clean but I still check nulls first
print("Null check:")
sdf.select([F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in sdf.columns]).show()

#StringIndexer converts species string to numeric label
#setosa=0.0,versicolor=1.0,virginica=2.0
if "label" in sdf.columns:
    sdf=sdf.drop("label")
lbl_idx=StringIndexer(inputCol="species",outputCol="label")
lbl_mod=lbl_idx.fit(sdf)
sdf=lbl_mod.transform(sdf)

#Combine 4 numeric features into one features column
#MLlib needs features in a single column
feat_cols=["sepal length (cm)","sepal width (cm)",
    "petal length (cm)","petal width (cm)"]

if "features" in sdf.columns:
    sdf=sdf.drop("features")
asm=VectorAssembler(inputCols=feat_cols,outputCol="features")
sdf=asm.transform(sdf)

#Keep only what we need
sdf_clean=sdf.select("features","label")

print("After preprocessing:")
sdf_clean.show(5,truncate=False)

Null check:
+-----------------+----------------+-----------------+----------------+-------+
|sepal length (cm)|sepal width (cm)|petal length (cm)|petal width (cm)|species|
+-----------------+----------------+-----------------+----------------+-------+
|                0|               0|                0|               0|      0|
+-----------------+----------------+-----------------+----------------+-------+

After preprocessing:
+-----------------+-----+
|features         |label|
+-----------------+-----+
|[5.1,3.5,1.4,0.2]|0.0  |
|[4.9,3.0,1.4,0.2]|0.0  |
|[4.7,3.2,1.3,0.2]|0.0  |
|[4.6,3.1,1.5,0.2]|0.0  |
|[5.0,3.6,1.4,0.2]|0.0  |
+-----------------+-----+
only showing top 5 rows


## Preprocessing Notes
Null check came back all zeros, nothing to fix.

StringIndexer mapped species to numbers:

setosa     -- 0.0

versicolor -- 1.0

virginica  -- 2.0

VectorAssembler packs all 4 feature columns
into one vector column. MLlib needs this
format to run any model.

In [ ]:
#6.Train test split 80:20
#seed=1234 so results are the same every run
train,test=sdf_clean.randomSplit([0.8, 0.2],seed=1234)

print("Train size:",train.count())
print("Test size :",test.count())

Train size: 113
Test size : 37


## Train Test Split

80% train, 20% test, seed=1234 so results
are the same every time I run it.

- Training: 113 rows
- Testing : 37 rows

In [ ]:
#7.Evaluation function
#reuse this for all 3 models instead of repeating code
def check_score(preds, name):
    ev_acc=MulticlassClassificationEvaluator(labelCol="label",
           predictionCol="prediction",metricName="accuracy")
    ev_f1=MulticlassClassificationEvaluator(labelCol="label",
           predictionCol="prediction",metricName="f1")
    ev_p=MulticlassClassificationEvaluator(labelCol="label",
           predictionCol="prediction",metricName="weightedPrecision")
    ev_r=MulticlassClassificationEvaluator(labelCol="label",
           predictionCol="prediction",metricName="weightedRecall")

    acc=ev_acc.evaluate(preds)
    f1=ev_f1.evaluate(preds)
    p=ev_p.evaluate(preds)
    r=ev_r.evaluate(preds)

    print("Model:",name)
    print("Accuracy:",round(acc, 4))
    print("Precision:",round(p, 4))
    print("Recall:",round(r, 4))
    print("F1:",round(f1, 4))
    return {"model": name, "accuracy": acc,
            "precision": p, "recall": r, "f1": f1}

In [ ]:
#8.Model 1：Logistic Regression
print("Running..")

lr=LogisticRegression(featuresCol="features",labelCol="label")

pg_lr=ParamGridBuilder() \
    .addGrid(lr.regParam, [0.0, 0.1, 0.3]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .addGrid(lr.maxIter, [20, 50]) \
    .build()

#5-fold CV is standard, gives more reliable estimate
#than 3-fold especially on small datasets
cv_lr=CrossValidator(estimator=lr,estimatorParamMaps=pg_lr,
      evaluator=MulticlassClassificationEvaluator(
        labelCol="label",predictionCol="prediction",
        metricName="accuracy"),numFolds=5,seed=1234)

cv_lr_mod=cv_lr.fit(train)
best_lr=cv_lr_mod.bestModel

#Print best params found by grid search
print("Best LR params:")
print("regParam:",best_lr._java_obj.getRegParam())
print("elasticNetParam:",best_lr._java_obj.getElasticNetParam())
print("maxIter:",best_lr._java_obj.getMaxIter())

#Best CV score during training
print("Best CV score:",round(max(cv_lr_mod.avgMetrics),4))

preds_lr=best_lr.transform(test)
preds_lr.select("features","label","prediction").show(10,truncate=False)

res_lr=check_score(preds_lr,"Logistic Regression")

Running..
Best LR params:
regParam: 0.0
elasticNetParam: 0.0
maxIter: 20
Best CV score: 0.9638
+-----------------+-----+----------+
|features         |label|prediction|
+-----------------+-----+----------+
|[4.4,2.9,1.4,0.2]|0.0  |0.0       |
|[4.5,2.3,1.3,0.3]|0.0  |1.0       |
|[5.0,2.0,3.5,1.0]|1.0  |1.0       |
|[5.0,3.3,1.4,0.2]|0.0  |0.0       |
|[5.0,3.4,1.5,0.2]|0.0  |0.0       |
|[5.0,3.4,1.6,0.4]|0.0  |0.0       |
|[5.0,3.5,1.3,0.3]|0.0  |0.0       |
|[5.1,3.8,1.6,0.2]|0.0  |0.0       |
|[5.4,3.7,1.5,0.2]|0.0  |0.0       |
|[5.4,3.9,1.7,0.4]|0.0  |0.0       |
+-----------------+-----+----------+
only showing top 10 rows
Model: Logistic Regression
Accuracy: 0.9459
Precision: 0.9543
Recall: 0.9459
F1: 0.9471


## Logistic Regression Results
Accuracy : 0.9459

Precision: 0.9543

Recall   : 0.9459

F1       : 0.9471

Best params: regParam=0.0, elasticNetParam=0.0, maxIter=20   
Best CV score: 0.9638

When I saw regParam=0.0 I thought that was weird.
Zero regularisation means the model is basically
running with no safety net at all. But then I
realised Iris is so clean that the model does
not need any regularisation to avoid overfitting.
It is extremely confident about this data and
that confidence turns out to be justified.

Grid search tested 18 combinations (3×3×2),
5-fold CV = 90 total fits.

CV score 0.9638, test score 0.9459.
Small gap. 37 test samples is not a lot
so I did not read too much into it.

In [ ]:
#9.Model 2：Decision Tree
print("Running..")

dt=DecisionTreeClassifier(featuresCol="features",labelCol="label",seed=1234)

pg_dt=ParamGridBuilder() \
    .addGrid(dt.maxDepth, [2, 3, 5]) \
    .addGrid(dt.minInstancesPerNode, [1, 2, 3]) \
    .build()

cv_dt=CrossValidator(estimator=dt,estimatorParamMaps=pg_dt,
    evaluator=MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction",
        metricName="accuracy"),numFolds=5,seed=1234)

cv_dt_mod=cv_dt.fit(train)
best_dt=cv_dt_mod.bestModel

#Print best params
print("Best DT params:")
print("maxDepth:",best_dt.depth)
print("minInstancesPerNode :",best_dt._java_obj.getMinInstancesPerNode())

print("Best CV score:",round(max(cv_dt_mod.avgMetrics),4))
preds_dt=best_dt.transform(test)
preds_dt.select("features","label","prediction").show(10,truncate=False)

res_dt=check_score(preds_dt,"Decision Tree")

Running..
Best DT params:
maxDepth: 2
minInstancesPerNode : 1
Best CV score: 0.9617
+-----------------+-----+----------+
|features         |label|prediction|
+-----------------+-----+----------+
|[4.4,2.9,1.4,0.2]|0.0  |0.0       |
|[4.5,2.3,1.3,0.3]|0.0  |0.0       |
|[5.0,2.0,3.5,1.0]|1.0  |1.0       |
|[5.0,3.3,1.4,0.2]|0.0  |0.0       |
|[5.0,3.4,1.5,0.2]|0.0  |0.0       |
|[5.0,3.4,1.6,0.4]|0.0  |0.0       |
|[5.0,3.5,1.3,0.3]|0.0  |0.0       |
|[5.1,3.8,1.6,0.2]|0.0  |0.0       |
|[5.4,3.7,1.5,0.2]|0.0  |0.0       |
|[5.4,3.9,1.7,0.4]|0.0  |0.0       |
+-----------------+-----+----------+
only showing top 10 rows
Model: Decision Tree
Accuracy: 0.9459
Precision: 0.9543
Recall: 0.9459
F1: 0.9463


## Decision Tree Results
Accuracy : 0.9459

Precision: 0.9543

Recall: 0.9459

F1: 0.9463

Best params: maxDepth=2, minInstancesPerNode=1  
Best CV score: 0.9617

Grid search picked maxDepth=2.
Honestly two splits and it is basically done.
Iris is that easy to separate.

Grid search tested 9 combinations (3×3),
5-fold CV = 45 total fits.

F1 is 0.9463 vs LR 0.9471. Tiny gap but LR
handled the one borderline case slightly better.

In [ ]:
#10.Model 3：Random Forest
print("Running..")

rf=RandomForestClassifier(featuresCol="features",labelCol="label",seed=1234)

pg_rf=ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20, 50]) \
    .addGrid(rf.maxDepth, [3, 5, 7]) \
    .build()

cv_rf=CrossValidator(estimator=rf,estimatorParamMaps=pg_rf,
    evaluator=MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction",
        metricName="accuracy"),numFolds=5,seed=1234)

cv_rf_mod=cv_rf.fit(train)
best_rf=cv_rf_mod.bestModel

# Print best params
print("Best RF params:")
print("numTrees:",best_rf.getNumTrees)
print("maxDepth:",best_rf.getMaxDepth())

print("Best CV score:",round(max(cv_rf_mod.avgMetrics),4))

preds_rf=best_rf.transform(test)
preds_rf.select("features","label","prediction").show(10,truncate=False)

res_rf=check_score(preds_rf,"Random Forest")

Running..
Best RF params:
numTrees: 20
maxDepth: 3
Best CV score: 0.9578
+-----------------+-----+----------+
|features         |label|prediction|
+-----------------+-----+----------+
|[4.4,2.9,1.4,0.2]|0.0  |0.0       |
|[4.5,2.3,1.3,0.3]|0.0  |0.0       |
|[5.0,2.0,3.5,1.0]|1.0  |1.0       |
|[5.0,3.3,1.4,0.2]|0.0  |0.0       |
|[5.0,3.4,1.5,0.2]|0.0  |0.0       |
|[5.0,3.4,1.6,0.4]|0.0  |0.0       |
|[5.0,3.5,1.3,0.3]|0.0  |0.0       |
|[5.1,3.8,1.6,0.2]|0.0  |0.0       |
|[5.4,3.7,1.5,0.2]|0.0  |0.0       |
|[5.4,3.9,1.7,0.4]|0.0  |0.0       |
+-----------------+-----+----------+
only showing top 10 rows
Model: Random Forest
Accuracy: 0.9459
Precision: 0.9543
Recall: 0.9459
F1: 0.9463


## Random Forest Results
Accuracy : 0.9459

Precision: 0.9543

Recall   : 0.9459

F1       : 0.9463

Best params: numTrees=20, maxDepth=3  
Best CV score: 0.9578

RF builds 20 trees and takes a majority vote.
Usually RF does better than one tree
but here the data is clean enough that
20 trees or 50 trees it does not matter.

Grid search tested 6 combinations (3×2),
5-fold CV = 30 total fits.

In [ ]:
#11.Compare all 3 models
rows=[Row(**res_lr),Row(**res_dt),Row(**res_rf)]
cmp=spark.createDataFrame(rows)

print("Model comparison:")
cmp.select("model",
    F.round("accuracy",4).alias("accuracy"),
    F.round("precision",4).alias("precision"),
    F.round("recall",4).alias("recall"),
    F.round("f1",4).alias("f1")
    ).orderBy(F.desc("accuracy"), F.desc("f1")).show(truncate=False)

Model comparison:
+-------------------+--------+---------+------+------+
|model              |accuracy|precision|recall|f1    |
+-------------------+--------+---------+------+------+
|Logistic Regression|0.9459  |0.9543   |0.9459|0.9471|
|Decision Tree      |0.9459  |0.9543   |0.9459|0.9463|
|Random Forest      |0.9459  |0.9543   |0.9459|0.9463|
+-------------------+--------+---------+------+------+



## Comparison Table
| Model               | Accuracy | Precision | Recall | F1     |
|---------------------|----------|-----------|--------|--------|
| Logistic Regression | 0.9459   | 0.9543    | 0.9459 | 0.9471 |
| Decision Tree       | 0.9459   | 0.9543    | 0.9459 | 0.9463 |
| Random Forest       | 0.9459   | 0.9543    | 0.9459 | 0.9463 |

When I saw all three hitting exactly 0.9459
I went back and checked my variable names
and the confusion matrix because I thought
something was reusing the same predictions.
Nothing was wrong. They genuinely scored
the same.

With only 37 test samples and two features
carrying most of the signal, all three models
end up making the same calls. Since all three models hit the same accuracy,
they likely made the same calls on the
borderline samples too.

LR wins on F1 at 0.9471 vs 0.9463 so it
is the best model this run.

Logistic Regression
- Strength: Fast, good on linearly separable
  data, coefficients are easy to read
- Limitation: Linear boundary only,
  fails on complex patterns

Decision Tree
- Strength: Every decision is traceable,
  easiest to explain
- Limitation: Can overfit without
  proper depth limit

Random Forest
- Strength: More stable than one tree,
  gives feature importance output
- Limitation: Slower to train, harder
  to explain individual predictions

In [ ]:
#12. Pick best model automatically
#I let the results decide, not hardcoded
top_row=cmp.orderBy(F.desc("accuracy"),F.desc("f1")).first()
top_name=top_row["model"]
print("Best model:",top_name)

if top_name=="Logistic Regression":
    top_preds=preds_lr
elif top_name=="Decision Tree":
    top_preds=preds_dt
else:
    top_preds=preds_rf

Best model: Logistic Regression


In [ ]:
#13.Final predictions from best model
print("Final predictions:")
top_preds.select("features","label","prediction").show(20,truncate=False)

print("Label vs prediction summary:")
top_preds.groupBy("label", "prediction") \
    .count() \
    .orderBy("label", "prediction") \
    .show()

#Show only the wrong ones
print("Misclassified samples:")
top_preds.filter(F.col("label") != F.col("prediction")
      ).select("features", "label", "prediction").show()

Final predictions:
+-----------------+-----+----------+
|features         |label|prediction|
+-----------------+-----+----------+
|[4.4,2.9,1.4,0.2]|0.0  |0.0       |
|[4.5,2.3,1.3,0.3]|0.0  |1.0       |
|[5.0,2.0,3.5,1.0]|1.0  |1.0       |
|[5.0,3.3,1.4,0.2]|0.0  |0.0       |
|[5.0,3.4,1.5,0.2]|0.0  |0.0       |
|[5.0,3.4,1.6,0.4]|0.0  |0.0       |
|[5.0,3.5,1.3,0.3]|0.0  |0.0       |
|[5.1,3.8,1.6,0.2]|0.0  |0.0       |
|[5.4,3.7,1.5,0.2]|0.0  |0.0       |
|[5.4,3.9,1.7,0.4]|0.0  |0.0       |
|[5.6,2.9,3.6,1.3]|1.0  |1.0       |
|[5.7,4.4,1.5,0.4]|0.0  |0.0       |
|[5.8,2.7,4.1,1.0]|1.0  |1.0       |
|[5.8,4.0,1.2,0.2]|0.0  |0.0       |
|[6.1,2.8,4.7,1.2]|1.0  |1.0       |
|[6.3,2.5,4.9,1.5]|1.0  |1.0       |
|[6.6,2.9,4.6,1.3]|1.0  |1.0       |
|[6.7,3.1,4.4,1.4]|1.0  |1.0       |
|[5.7,2.5,5.0,2.0]|2.0  |2.0       |
|[5.7,3.0,4.2,1.2]|1.0  |1.0       |
+-----------------+-----+----------+
only showing top 20 rows
Label vs prediction summary:
+-----+----------+-----+
|label|predict

## Confusion Matrix (Best Model: Logistic Regression)
| Actual \ Predicted | setosa | versicolor | virginica |
|--------------------|------------|----------------|---------------|
| 0.0 setosa         | 10         | 1              | 0             |
| 1.0 versicolor     | 0          | 11             | 0             |
| 2.0 virginica      | 0          | 1              | 14            |

Total wrong: 2 out of 37

Misclassified samples:
- [4.5,2.3,1.3,0.3] setosa predicted as versicolor
- [6.0,3.0,4.8,1.8] virginica predicted as versicolor

Both mistakes land on the versicolor boundary.
That region is where LR got confused.
Since all three models hit the same accuracy,
they likely struggled with the same samples.

In [ ]:
#14.Feature importance from Random Forest
print("Random Forest feature importance:")
for name, score in zip(feat_cols, best_rf.featureImportances):
    print(f"  {name} : {round(float(score), 4)}")

Random Forest feature importance:
  sepal length (cm) : 0.0973
  sepal width (cm) : 0.0013
  petal length (cm) : 0.4701
  petal width (cm) : 0.4312


## Feature Importance (Random Forest)

| Feature          | Importance |
|------------------|------------|
| sepal length (cm)| 0.0973     |
| sepal width (cm) | 0.0013     |
| petal length (cm)| 0.4701     |
| petal width (cm) | 0.4312     |

Petal length and petal width together are
over 90% of the total importance.

Sepal width at 0.0013 is almost zero.
I did think about dropping it and rerunning
to see what happens. Decided to leave it in
since the task did not ask for feature selection
and I was not sure if removing it would
actually change anything or just complicate things.

That is probably also why all three models
ended up at the same number. When two features
carry everything, there is not much
for models to disagree on.

In [ ]:
#15.Done
spark.stop()
print("Done, Spark stopped")

Done, Spark stopped


## Overall Conclusion

Best model: Logistic Regression (F1 = 0.9471)

Honestly the hardest part of this assignment
was getting Spark to run correctly. When
something breaks in Spark the error messages
are not always clear and debugging takes time.
Once the pipeline was working the modelling
part was more straightforward.

The results themselves made sense once I
looked at feature importance. Petal length
and petal width carry almost everything.
Any model that finds those two features
will get roughly the same accuracy which
is exactly what happened.

The two wrong predictions from LR both sit on
the versicolor boundary. Since all three models
hit the same accuracy and F1, they likely
made the same mistakes too. That region just
seems to be the hardest part of this dataset
regardless of which model you use.

Things I would try if I continued this:
- Remove sepal width since it contributes
  almost nothing and see if it matters
- Run on a noisier dataset where model
  differences would actually show up